# Stage 1b — Merge (standalone)

Your Stage-1 run **trained successfully** — it only failed on the final merge
(the torchao version quirk). This notebook loads that trained adapter and
finishes the job: merge into the base model and push to the Hub. Use it when
you can't edit the original notebook.

### Two ways to reach the adapter (this notebook tries both)
1. **From the Kaggle output** — New notebook → **+ Add Input** → add your Stage-1
   training notebook. Its saved `/kaggle/working` mounts under `/kaggle/input/`.
   This gives the exact final adapter.
2. **From the Hub** — training pushed checkpoints to your `swa-gemma-cpt` repo as
   it ran, so the adapter is already there even if the Kaggle output isn't.

**Setup:** Accelerator **GPU T4 ×2**, Internet **ON**, add your `HF_TOKEN` secret,
then run top to bottom.

In [8]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [9]:
%%capture
# The only fix needed for the merge: remove the too-old torchao that makes PEFT raise.
!pip uninstall -y torchao

In [10]:
import torch, os, glob
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF auth OK | CUDA:", torch.cuda.is_available())

HF auth OK | CUDA: True


In [11]:
MODEL_ID       = "google/gemma-3-1b-it"
HF_USERNAME    = "Dev-Noble"                     # <-- EDIT (same as Stage 1)
HUB_CPT_CKPT   = f"{HF_USERNAME}/swa-gemma-cpt"         # where training pushed checkpoints
HUB_CPT_MERGED = f"{HF_USERNAME}/swa-gemma-cpt-merged"  # output of this notebook
assert HF_USERNAME != "YOUR_HF_USERNAME", "Set HF_USERNAME first."

## Locate the adapter

Tries the mounted Kaggle input first; if nothing is there, falls back to the
checkpoint on the Hub.

In [12]:
adapter_dir = None

# 1) Look in the mounted Kaggle notebook output. Every PEFT adapter dir has an
#    adapter_config.json, so we search for that regardless of the mount path.
hits = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
if hits:
    print("Adapter folders found under /kaggle/input:")
    for h in hits:
        print("  ", os.path.dirname(h))
    final = [h for h in hits if "final_adapter" in h]
    adapter_dir = os.path.dirname(final[0]) if final else os.path.dirname(hits[-1])
    print("\nUsing (from Kaggle input):", adapter_dir)

# 2) Fallback: pull the checkpoint you pushed to the Hub during training.
if adapter_dir is None:
    print("Nothing under /kaggle/input -> pulling last-checkpoint from the Hub...")
    from huggingface_hub import snapshot_download
    local = snapshot_download(HUB_CPT_CKPT, allow_patterns="last-checkpoint/*")
    adapter_dir = os.path.join(local, "last-checkpoint")
    print("Using (from Hub):", adapter_dir)

assert os.path.exists(os.path.join(adapter_dir, "adapter_config.json")), \
    "No adapter_config.json in adapter_dir — check the path."

Nothing under /kaggle/input -> pulling last-checkpoint from the Hub...


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Using (from Hub): /root/.cache/huggingface/hub/models--Dev-Noble--swa-gemma-cpt/snapshots/c9771ed9f639adc945a1570ca418a4f779c1769b/last-checkpoint


## Merge and push

In [13]:
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()

# Tokenizer: training saved it alongside the adapter; fall back to base if absent.
tok_src = adapter_dir if os.path.exists(
    os.path.join(adapter_dir, "tokenizer_config.json")) else MODEL_ID
tok = AutoTokenizer.from_pretrained(tok_src)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

merged.push_to_hub(HUB_CPT_MERGED, private=True)
tok.push_to_hub(HUB_CPT_MERGED)
print("Merged CPT model pushed to:", HUB_CPT_MERGED)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Merged CPT model pushed to: Dev-Noble/swa-gemma-cpt-merged


## Smoke test — does it produce Swahili?

In [14]:
merged.config.use_cache = True; merged.eval()

def translate(en, n=64):
    p = ("<start_of_turn>user\nTranslate the following English sentence to Swahili.\n"
         f"English: {en}\nSwahili:<end_of_turn>\n<start_of_turn>model\n")
    enc = tok(p, return_tensors="pt").to(merged.device)
    out = merged.generate(**enc, max_new_tokens=n, num_beams=5, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for s in ["The children are playing outside.",
          "The government will build a new hospital next year."]:
    print(s, "\n  ->", translate(s), "\n")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The children are playing outside. 
  -> Here's the translation of the English sentence "The children are playing outside" into Swahili:

**Miongoni mwa watoto wamecheza nje."**

Here's a breakdown of the translation:

*   **Miongoni mwa watoto:** "Among the children" 

The government will build a new hospital next year. 
  -> Here's the translation of the English sentence to Swahili:

**Serikali itakuwa imeweka hospitali mpya mwaka huu. (Serikali itakuwa imeweka hospitali mpya mwaka huu.)**

Here's a breakdown of the translation:

*   **Serikali 

